# Yankee Stadium Game & Weather Dataset (2015-2025)

This notebook builds a comprehensive dataset where each row represents a regular season game played at Yankee Stadium. It combines:
1. **Game statistics** from `master_data.csv` (pre-aggregated Statcast data)
2. **Weather data** from Open-Meteo hourly observations
3. **Wind projections** onto outfield vectors (CF, LCF, RCF)

All batting/pitching statistics are **both teams combined** to capture the full park-environment effect.

## Section 0: Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import requests
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# === STADIUM CONFIGURATION ===
STADIUM_NAME = 'Yankee Stadium'
HOME_TEAM = 'NYY'
SEASONS = range(2015, 2026)  # 2015 through 2025
TIMEZONE = 'America/New_York'

# Coordinates
STADIUM_LAT = 40.829682
STADIUM_LON = -73.926300

# Outfield directions (degrees from north)
CF_DIR = 67.5    # Center field: ENE
LCF_DIR = 47.5   # Left-center field: NE
RCF_DIR = 87.5   # Right-center field: E

# Output file
OUTPUT_FILE = 'yankee_data_2015.csv'

print(f"Configuration: {STADIUM_NAME}")
print(f"Home team: {HOME_TEAM}")
print(f"Seasons: {list(SEASONS)}")
print(f"Timezone: {TIMEZONE}")

/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Configuration: Yankee Stadium
Home team: NYY
Seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Timezone: America/New_York


## Section 1: Load & Filter Master Data

Read game-level statistics from `master_data.csv` and filter to Yankee Stadium home games.

In [2]:
# Load master dataset
master = pd.read_csv(os.path.join('..', 'Final Datasets', 'master_data.csv'))
print(f"Master dataset: {len(master)} total games")

# Filter to this stadium's home games and season range
games = master[
    (master['home_team'] == HOME_TEAM) &
    (master['season'].isin(SEASONS))
].copy()

# Convert game_start_utc to local timezone
games['game_start'] = (
    pd.to_datetime(games['game_start_utc'], utc=True)
    .dt.tz_convert(TIMEZONE)
    .dt.tz_localize(None)  # Remove timezone info for clean processing
)
games['start_hour'] = games['game_start'].dt.hour
games['game_date'] = pd.to_datetime(games['game_date'])

# Drop master-only columns not needed in final output
games = games.drop(columns=['home_team', 'game_start_utc'])

games = games.sort_values('game_date').reset_index(drop=True)

print(f"\n{STADIUM_NAME} games: {len(games)}")
print(f"Seasons: {sorted(games['season'].unique())}")
print(f"\nGames per season:")
print(games.groupby('season')['game_pk'].count())
print(f"\nSample:")
print(games.head(3))

Master dataset: 25155 total games

Yankee Stadium games: 839
Seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Games per season:
season
2015    81
2016    80
2017    81
2018    81
2019    81
2020    30
2021    81
2022    81
2023    81
2024    81
2025    81
Name: game_pk, dtype: int64

Sample:
   game_pk  game_date  season away_team  home_runs_scored  away_runs_scored  total_runs  home_runs_hit  strikeouts  walks  hits  total_pitches  avg_exit_velocity  n_barrels  n_bbe  barrel_rate  \
0   413663 2015-04-06    2015       TOR                 1                 6           7              3          17      8     9            297               87.7          3     46       0.0652   
1   413685 2015-04-08    2015       TOR                 4                 3           7              0          14      7    14            289               84.4          1     49       0.0204   
2   413695 2015-04-09    2015       TOR                 3                 6           9    

## Section 2: Pull Weather Data (Open-Meteo)

Use Open-Meteo to pull hourly weather data for each season, then average over the 3 hours following each game's start time.

In [3]:
# Using Open-Meteo Historical Weather API (free, no key required)
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_PARAMS = "temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m"

# Test fetch to confirm API is reachable
test_resp = requests.get(OPEN_METEO_URL, params={
    'latitude': STADIUM_LAT,
    'longitude': STADIUM_LON,
    'start_date': '2023-07-01',
    'end_date': '2023-07-02',
    'hourly': HOURLY_PARAMS,
    'timezone': TIMEZONE,
})

if test_resp.status_code == 200:
    test_data = test_resp.json()
    n_hours = len(test_data['hourly']['time'])
    print(f"Open-Meteo API test (Jul 1-2 2023): {n_hours} hourly records - OK")
    print(f"Sample time: {test_data['hourly']['time'][12]}")
    print(f"Sample temp: {test_data['hourly']['temperature_2m'][12]}°C")
    print(f"Sample wind: {test_data['hourly']['wind_speed_10m'][12]} km/h from {test_data['hourly']['wind_direction_10m'][12]}°")
else:
    print(f"ERROR: Open-Meteo API returned {test_resp.status_code}")
    print(test_resp.text)

Open-Meteo API test (Jul 1-2 2023): 48 hourly records - OK
Sample time: 2023-07-01T12:00
Sample temp: 27.0°C
Sample wind: 20.2 km/h from 164°


In [4]:
def fetch_season_weather(year):
    """Fetch hourly weather for a full season from Open-Meteo."""
    resp = requests.get(OPEN_METEO_URL, params={
        'latitude': STADIUM_LAT,
        'longitude': STADIUM_LON,
        'start_date': f'{year}-03-01',
        'end_date': f'{year}-11-30',
        'hourly': HOURLY_PARAMS,
        'timezone': TIMEZONE,
    })
    resp.raise_for_status()
    hourly = resp.json()['hourly']
    
    df = pd.DataFrame({
        'temp': hourly['temperature_2m'],
        'rhum': hourly['relative_humidity_2m'],
        'pres': hourly['surface_pressure'],
        'prcp': hourly['precipitation'],
        'wspd': hourly['wind_speed_10m'],
        'wdir': hourly['wind_direction_10m'],
    }, index=pd.to_datetime(hourly['time']))
    
    return df


def get_game_weather(game_start_dt, hourly_df):
    """
    Average weather over the 3 hours following game start.
    game_start_dt: datetime (local time, rounded to hour)
    hourly_df: DataFrame with hourly weather, index is naive local time
    """
    start = game_start_dt
    end = start + timedelta(hours=2)  # 3 hourly obs: start, +1h, +2h
    
    window = hourly_df.loc[start:end]
    
    if len(window) == 0:
        return pd.Series({
            'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
            'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
        })
    
    result = {
        'temp_c': window['temp'].mean(),
        'rhum': window['rhum'].mean(),
        'pres': window['pres'].mean(),
        'prcp': window['prcp'].sum(),   # Precipitation SUMMED (cumulative quantity)
        'wspd': window['wspd'].mean(),
    }
    
    # Wind direction: circular mean to handle 0/360 boundary
    wdir_vals = window['wdir'].dropna()
    if len(wdir_vals) > 0:
        wdir_rad = np.radians(wdir_vals)
        mean_sin = np.sin(wdir_rad).mean()
        mean_cos = np.cos(wdir_rad).mean()
        result['wdir'] = np.degrees(np.arctan2(mean_sin, mean_cos)) % 360
    else:
        result['wdir'] = np.nan
    
    return pd.Series(result)


# Pull weather season by season
weather_records = []

for year in SEASONS:
    print(f"Pulling weather for {year}...")
    
    try:
        hourly_df = fetch_season_weather(year)
    except Exception as e:
        print(f"  WARNING: Failed for {year}: {e}")
        season_games = games[games['season'] == year]
        for idx, game in season_games.iterrows():
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
        continue
    
    print(f"  {year}: {len(hourly_df)} hourly records")
    
    season_games = games[games['season'] == year]
    for idx, game in season_games.iterrows():
        if pd.isna(game['game_start']):
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
            continue
        
        game_hour = game['game_start'].replace(minute=0, second=0, microsecond=0)
        wx = get_game_weather(game_hour, hourly_df)
        wx['game_pk'] = game['game_pk']
        weather_records.append(wx.to_dict())

weather_df = pd.DataFrame(weather_records)
weather_df['game_pk'] = weather_df['game_pk'].astype('Int64')
print(f"\nWeather records: {len(weather_df)}")
print(f"Missing temp data: {weather_df['temp_c'].isna().sum()}")
print(weather_df.head(3))

Pulling weather for 2015...
  2015: 6600 hourly records
Pulling weather for 2016...
  2016: 6600 hourly records
Pulling weather for 2017...
  2017: 6600 hourly records
Pulling weather for 2018...
  2018: 6600 hourly records
Pulling weather for 2019...
  2019: 6600 hourly records
Pulling weather for 2020...
  2020: 6600 hourly records
Pulling weather for 2021...
  2021: 6600 hourly records
Pulling weather for 2022...
  2022: 6600 hourly records
Pulling weather for 2023...
  2023: 6600 hourly records
Pulling weather for 2024...
  2024: 6600 hourly records
Pulling weather for 2025...
  2025: 6600 hourly records

Weather records: 839
Missing temp data: 0
      temp_c       rhum         pres  prcp       wspd        wdir  game_pk
0  15.900000  56.333333  1021.866667   0.0  10.533333  168.333093   413663
1   4.266667  74.000000  1024.166667   0.0  16.200000   72.001527   413685
2   4.233333  83.666667  1024.766667   0.0   9.466667   83.338055   413695


## Section 3: Wind Direction Bucketing & Outfield Projections

**Wind direction bucketing**: 8 compass directions (N, NE, E, SE, S, SW, W, NW).

**Wind projections**: Project wind onto vectors from home plate to center field (CF), left-center field (LCF), and right-center field (RCF). Positive = blowing out, negative = blowing in.

Yankee Stadium outfield directions (degrees from north):
- Center field: ~67.5° (ENE)
- Left-center field: ~47.5° (NE)
- Right-center field: ~87.5° (E)

**Important**: Weather APIs report wind direction as the direction wind blows **FROM**. We must convert to the direction it blows **TO** before projecting.

In [5]:
# Merge weather into game data
games_full = games.merge(weather_df, on='game_pk', how='left')

# --- Wind direction bucketing ---
def bucket_wind_dir(deg):
    """Bucket wind direction (degrees) into 8 compass directions."""
    if pd.isna(deg):
        return np.nan
    buckets = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    idx = int(((deg + 22.5) % 360) / 45)
    return buckets[idx]

games_full['wind_dir_bucket'] = games_full['wdir'].apply(bucket_wind_dir)

# --- Wind projections onto outfield vectors ---
def compute_wind_projection(wdir, wspd, outfield_dir):
    """
    Project wind onto an outfield direction vector.
    
    wdir: direction wind blows FROM (meteorological convention, degrees)
    wspd: wind speed (km/h)
    outfield_dir: compass bearing from home plate to outfield (degrees from north)
    
    Returns: positive = blowing OUT toward outfield, negative = blowing IN
    """
    if pd.isna(wdir) or pd.isna(wspd):
        return np.nan
    # Wind blows FROM wdir, so it travels TOWARD (wdir + 180)
    wind_toward = (wdir + 180) % 360
    # Project onto outfield direction
    angle_diff = wind_toward - outfield_dir
    return wspd * np.cos(np.radians(angle_diff))

games_full['wind_cf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], CF_DIR), axis=1
)
games_full['wind_lcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], LCF_DIR), axis=1
)
games_full['wind_rcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], RCF_DIR), axis=1
)

print("Wind projection summary (positive = blowing out, negative = blowing in):")
print(games_full[['wind_cf', 'wind_lcf', 'wind_rcf']].describe())

Wind projection summary (positive = blowing out, negative = blowing in):
          wind_cf    wind_lcf    wind_rcf
count  839.000000  839.000000  839.000000
mean     2.098299    2.632483    1.311030
std      8.471416    8.514571    8.792594
min    -30.696752  -30.984640  -28.692776
25%     -2.783032   -1.708088   -4.152408
50%      2.907262    3.663359    0.644548
75%      7.483372    7.914933    7.546729
max     29.307534   26.992560   31.761057


## Section 4: Final Assembly

Convert units, order columns, and round to sensible precision.

In [6]:
# Unit conversions
games_full['temp_f'] = games_full['temp_c'] * 9/5 + 32
games_full['wspd_mph'] = games_full['wspd'] * 0.621371

# Final column order
final_columns = [
    # Game identification
    'game_pk', 'game_date', 'season', 'away_team', 'game_start', 'start_hour',
    # Scoring
    'home_runs_scored', 'away_runs_scored', 'total_runs',
    # Batting stats (both teams combined)
    'home_runs_hit', 'strikeouts', 'walks', 'hits',
    'total_pitches', 'avg_exit_velocity',
    'n_barrels', 'n_bbe', 'barrel_rate', 'hr_h_ratio',
    # Weather
    'temp_f', 'temp_c', 'rhum', 'pres', 'prcp',
    'wspd', 'wspd_mph', 'wdir', 'wind_dir_bucket',
    # Wind projections
    'wind_cf', 'wind_lcf', 'wind_rcf',
]

yankee_data = games_full[final_columns].copy()
yankee_data = yankee_data.sort_values('game_date').reset_index(drop=True)

# Round floating point columns
round_map = {
    'avg_exit_velocity': 1, 'barrel_rate': 4, 'hr_h_ratio': 4,
    'temp_f': 1, 'temp_c': 1, 'rhum': 1, 'pres': 1, 'prcp': 2,
    'wspd': 1, 'wspd_mph': 1, 'wdir': 1,
    'wind_cf': 2, 'wind_lcf': 2, 'wind_rcf': 2,
}
for col, decimals in round_map.items():
    yankee_data[col] = yankee_data[col].round(decimals)

print(f"Final dataset: {yankee_data.shape[0]} rows x {yankee_data.shape[1]} columns")

Final dataset: 839 rows x 31 columns


## Section 5: Validation

Verify row counts per season, check for nulls, and sanity-check summary statistics.

In [7]:
print("=" * 70)
print("VALIDATION REPORT")
print("=" * 70)

# 1. Row counts per season
print("\n--- Games per Season ---")
season_counts = yankee_data.groupby('season').size()
for year, count in season_counts.items():
    if year == 2020:
        expected = (25, 35)  # COVID shortened season
    else:
        expected = (75, 100)  # Normal: ~81 home games (wider range for doubleheaders)
    status = "OK" if expected[0] <= count <= expected[1] else "WARNING"
    print(f"  {year}: {count} games [{status}] (expected {expected[0]}-{expected[1]})")
print(f"  TOTAL: {len(yankee_data)} games")

# 2. Null check
print("\n--- Null Counts ---")
key_cols = ['total_runs', 'home_runs_hit', 'strikeouts', 'walks',
            'total_pitches', 'avg_exit_velocity', 'barrel_rate',
            'temp_f', 'wspd', 'wdir', 'wind_cf', 'game_start']
for col in key_cols:
    n_null = yankee_data[col].isna().sum()
    pct = 100 * n_null / len(yankee_data)
    status = "OK" if pct < 5 else "WARNING"
    print(f"  {col}: {n_null} nulls ({pct:.1f}%) [{status}]")

# 3. Summary statistics sanity checks
print("\n--- Sanity Checks ---")
checks = [
    ('Avg total runs/game', yankee_data['total_runs'].mean(), '~8-10'),
    ('Avg HR/game', yankee_data['home_runs_hit'].mean(), '~2-3'),
    ('Avg K/game', yankee_data['strikeouts'].mean(), '~16-18'),
    ('Avg BB/game', yankee_data['walks'].mean(), '~6-7'),
    ('Avg exit velocity', yankee_data['avg_exit_velocity'].mean(), '~87-89 mph'),
    ('Avg barrel rate', yankee_data['barrel_rate'].mean(), '~0.06-0.08'),
    ('Avg game temp', yankee_data['temp_f'].mean(), '~60-70 F'),
    ('Min game temp', yankee_data['temp_f'].min(), '>30 F'),
    ('Max game temp', yankee_data['temp_f'].max(), '<105 F'),
    ('Avg wind speed (km/h)', yankee_data['wspd'].mean(), '~10-20 km/h'),
]
for label, val, expected in checks:
    print(f"  {label}: {val:.2f} (expected {expected})")

# 4. Full summary statistics
print("\n--- Summary Statistics ---")
print(yankee_data.describe().T[['mean', 'std', 'min', 'max']].to_string())

VALIDATION REPORT

--- Games per Season ---
  2015: 81 games [OK] (expected 75-100)
  2016: 80 games [OK] (expected 75-100)
  2017: 81 games [OK] (expected 75-100)
  2018: 81 games [OK] (expected 75-100)
  2019: 81 games [OK] (expected 75-100)
  2020: 30 games [OK] (expected 25-35)
  2021: 81 games [OK] (expected 75-100)
  2022: 81 games [OK] (expected 75-100)
  2023: 81 games [OK] (expected 75-100)
  2024: 81 games [OK] (expected 75-100)
  2025: 81 games [OK] (expected 75-100)
  TOTAL: 839 games

--- Null Counts ---
  total_runs: 0 nulls (0.0%) [OK]
  home_runs_hit: 0 nulls (0.0%) [OK]
  strikeouts: 0 nulls (0.0%) [OK]
  walks: 0 nulls (0.0%) [OK]
  total_pitches: 0 nulls (0.0%) [OK]
  avg_exit_velocity: 0 nulls (0.0%) [OK]
  barrel_rate: 0 nulls (0.0%) [OK]
  temp_f: 0 nulls (0.0%) [OK]
  wspd: 0 nulls (0.0%) [OK]
  wdir: 0 nulls (0.0%) [OK]
  wind_cf: 0 nulls (0.0%) [OK]
  game_start: 0 nulls (0.0%) [OK]

--- Sanity Checks ---
  Avg total runs/game: 9.06 (expected ~8-10)
  Avg HR/ga

In [8]:
# Print dataset header for inspection
print("\n--- First 10 Rows ---")
yankee_data.head(10)


--- First 10 Rows ---


,game_pk,game_date,season,away_team,game_start,start_hour,home_runs_scored,away_runs_scored,total_runs,home_runs_hit,strikeouts,walks,hits,total_pitches,avg_exit_velocity,n_barrels,n_bbe,barrel_rate,hr_h_ratio,temp_f,temp_c,rhum,pres,prcp,wspd,wspd_mph,wdir,wind_dir_bucket,wind_cf,wind_lcf,wind_rcf
0,413663,2015-04-06,2015,TOR,2015-04-06 13:05:00,13,1,6,7,3,17,8,9,297,87.7,3,46,0.0652,0.3333,60.6,15.9,56.3,1021.9,0.0,10.5,6.5,168.3,S,1.98,5.40,-1.68
1,413685,2015-04-08,2015,TOR,2015-04-08 19:05:00,19,4,3,7,0,14,7,14,289,84.4,1,49,0.0204,0.0000,39.7,4.3,74.0,1024.2,0.0,16.2,10.1,72.0,E,-16.15,-14.74,-15.61
2,413695,2015-04-09,2015,TOR,2015-04-09 19:05:00,19,3,6,9,3,20,4,16,294,84.0,3,47,0.0638,0.1875,39.6,4.2,83.7,1024.8,0.0,9.5,5.9,83.3,E,-9.11,-7.67,-9.44
3,413696,2015-04-10,2015,BOS,2015-04-10 19:05:00,19,5,6,11,3,28,14,32,628,89.0,2,113,0.0177,0.0938,55.5,13.1,95.0,1008.1,0.0,13.9,8.7,248.3,W,13.93,13.02,13.16
4,413711,2015-04-11,2015,BOS,2015-04-11 13:05:00,13,4,8,12,1,14,7,14,302,86.0,1,55,0.0182,0.0714,55.5,13.0,41.7,1016.2,0.0,19.6,12.2,301.7,NW,11.49,5.36,16.24
5,413726,2015-04-12,2015,BOS,2015-04-12 20:05:00,20,14,4,18,4,11,9,24,323,88.0,2,63,0.0317,0.1667,47.7,8.7,73.0,1025.9,0.0,9.5,5.9,193.3,S,5.56,7.86,2.59
6,413893,2015-04-24,2015,NYM,2015-04-24 19:05:00,19,6,1,7,3,14,2,16,255,85.9,3,53,0.0566,0.1875,40.9,5.0,50.3,1012.4,0.0,16.3,10.1,335.0,NW,0.71,-4.90,6.24
7,413908,2015-04-25,2015,NYM,2015-04-25 16:05:00,16,2,8,10,4,13,2,17,262,86.8,4,58,0.0690,0.2353,56.8,13.8,39.0,1005.1,0.0,6.1,3.8,238.3,SW,5.99,5.96,5.30
8,413923,2015-04-26,2015,NYM,2015-04-26 20:05:00,20,6,4,10,2,17,2,16,269,86.9,2,48,0.0417,0.1250,48.0,8.9,63.0,1004.7,0.0,12.4,7.7,319.0,NW,3.95,-0.33,7.74
9,413941,2015-04-27,2015,TB,2015-04-27 19:05:00,19,4,1,5,1,18,6,15,289,88.4,1,47,0.0213,0.0667,51.4,10.8,54.7,1005.5,0.0,15.5,9.7,311.7,NW,6.77,1.58,11.14


## Section 6: Save to CSV

In [9]:
# Save final dataset
output_dir = os.path.join('..', 'Final Datasets')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, OUTPUT_FILE)
yankee_data.to_csv(output_path, index=False)

print(f"Saved to: {os.path.abspath(output_path)}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Rows: {len(yankee_data)}, Columns: {len(yankee_data.columns)}")

# Verify roundtrip
verify = pd.read_csv(output_path)
assert verify.shape == yankee_data.shape, f"Shape mismatch: {verify.shape} vs {yankee_data.shape}"
print("\nSave & reload verification: PASSED")

Saved to: /Users/avabrown/Desktop/DATASCI 192A/Stadium Datasets/Final Datasets/yankee_data_2015.csv
File size: 125.8 KB
Rows: 839, Columns: 31

Save & reload verification: PASSED
